# ECB API & Norges Bank API - bronze
Valutakurser fra 2 API, lagret i hver sin tabell

## Libraries

In [ ]:
from pyspark.sql import SparkSession

# Initialize a local Spark Session
spark = SparkSession.builder \
    .appName("TestApp") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
from datetime import datetime
from datetime import timedelta
import io
import pandas as pd
from pyspark.sql import Window
import requests

## Variabler

In [ ]:
period_start = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
period_end = datetime.now().strftime("%Y-%m-%d")

base_url_ecb = "https://data-api.ecb.europa.eu/service/data/EXR/D.{pair}.SP00.A?format=csvdata&startPeriod={start}&endPeriod={end}"
base_url_nb = f"https://data.norges-bank.no/api/data/EXR/B.SEK+EUR+DKK+GBP+PLN.NOK.SP?format=csv&startPeriod={period_start}&endPeriod={period_end}&locale=en"

currency_pairs_ecb = {
"NOK to EUR": "NOK.EUR",
"DKK to EUR": "DKK.EUR",
"SEK to EUR": "SEK.EUR",
"GBP to EUR": "GBP.EUR",
"PLN to EUR": "PLN.EUR"
}

## Kode

In [ ]:
# norges bank api
try: 
    response = requests.get(base_url_nb)

    buffer = io.BytesIO(response.content)

    df_nb = pd.read_csv(buffer, sep = ";")

    df_nb.rename(columns={"Tenor": "Tenor_2"}, inplace=True)

    df_nb = spark.createDataFrame(df_nb) 
except Exception as e:
    print(f"Got en error while reading the NB API to dataframe: {e}. The API returned a response code of {response.status_code}")
    df_nb = None

In [ ]:
# ecb api
df_ecb = None

for name, pair in currency_pairs_ecb.items():

    url = base_url_ecb.format(pair=pair, start=period_start, end=period_end)

    response = requests.get(url)

    # read with pandas with using the bytesio as buffer
    temp_ecb = pd.read_csv(io.BytesIO(response.content), sep=",")

    # adding helper columns
    temp_ecb["conversion"] = name
    temp_ecb["pair_code"] = pair

    temp_ecb = spark.createDataFrame(temp_ecb) 

    # if spark df is not None append the new one in it
    df_ecb = temp_ecb if df_ecb is None else df_ecb.unionByName(temp_ecb)

   # if not temp_ecb.empty:
   #     df_ecb = pd.concat([df_ecb, temp_ecb])


# Silver
Reads the data from ECB and NB, unions them and saves to lake

In [ ]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

## Variables

In [ ]:
# ECB
# computing a id_calendar and id_calendar_month from the timeperiod column

# first converting the time_period to date
df_ecb = df_ecb.withColumn(
    "time_period", F.to_date(F.col("time_period"), "yyyy-MM-dd")
)

# creating the id_columns
df_ecb = df_ecb.withColumn(
    "id_date", F.date_format(F.col("time_period"), "yyyyMMdd")
    ).withColumn(
    "id_year_month", F.date_format(F.col("time_period"), "yyyyMM")
)

df_ecb = df_ecb.withColumn("source_api", F.lit("ECB")
)

# dict to rename columns, also used for ordering, order matters
cols_to_rename_ecb = {
"id_date": "id_date",
"id_year_month": "id_year_month",
"currency": "base_currency",
"currency_denom": "to_currency",
"obs_value": "conversion_rate",
"source_api": "source_api"
}

df_ecb = df_ecb.withColumnsRenamed(cols_to_rename_ecb)

col_order_ecb = [new_col for old_col, new_col in cols_to_rename_ecb.items()]

df_ecb = df_ecb.select(*col_order_ecb)

In [ ]:
# NB
# computing a id_calendar and id_calendar_month from the timeperiod column

# first converting the time_period to date
df_nb = df_nb.withColumn(
 "time_period", F.to_date(F.col("time_period"), "yyyy-MM-dd")
)

# creating the id_columns
df_nb = df_nb.withColumn("id_date", F.date_format(F.col("time_period"), "yyyyMMdd")
    ).withColumn("id_year_month", F.date_format(F.col("time_period"), "yyyyMM")
)

# nb might have hundreds as a multiplier for the selected currencies, dividing the rates and rounding it to 4 decimals
df_nb = df_nb.withColumn(
 "obs_value_adjusted",
 F.round(F.when(F.col("unit multiplier") == "Hundreds", F.col("obs_value") / 100.0)
 .otherwise(F.col("obs_value")), 4 # number of decimals
)
)

df_nb = df_nb.withColumn(
 "source_api", F.lit("NorgesBank")
)

cols_to_rename_nb = {
    "id_date": "id_date",
    "id_year_month": "id_year_month",
    "base_cur": "base_currency",
    "quote_cur": "to_currency",
    "obs_value_adjusted": "conversion_rate",
    "source_api": "source_api"
}

df_nb = df_nb.withColumnsRenamed(
 cols_to_rename_nb
)

col_order_nb = [new_col for old_col, new_col in cols_to_rename_nb.items()]

df_nb = df_nb.select(*col_order_nb)


# union the dataframes together
df_base = df_ecb.unionByName(df_nb)



# Gold
Creating 2 tables from the silver fact_currency, one contains daily averages, the other monthly averages"

## Kode

In [ ]:
## fact_currency_exchange_rate_daily

# the modifications to the daily table are the following: drop the year month column and fill any missing dates with the previous value
# filling the missing dates cannot be done before this step as we dont want them to affect the monthly averages
df_daily = df_base.drop(F.col("id_year_month"))

# we need to generate the full range of days to identify gaps
df_daily = df_daily.withColumn("id_date", F.to_date("id_date", "yyyyMMdd"))

min_date, max_date = df_daily.agg(F.min("id_date"), F.max("id_date")).first()

# full range of days, based on the min and max date from the source df
date_df = spark.createDataFrame([(min_date, max_date)], ["start", "end"]
).withColumn("id_date", F.explode(F.sequence(F.col("start"), F.col("end")))
).select("id_date")

currency_pairs = df_daily.select("base_currency", "to_currency", "source_api").distinct()

# full df with all the dates and currency pairs
full_df = date_df.crossJoin(currency_pairs)

# join to original data, now missing dates have nulls for conversion rates
joined_df = full_df.join(df_daily, ["id_date", "base_currency", "to_currency", "source_api"], "left")

# creating a window for filling the nulls
window = Window.partitionBy("base_currency", "to_currency", "source_api").orderBy("id_date")

# filling the nulls
filled_df = joined_df.withColumn(
 "conversion_rate",F.last("conversion_rate", ignorenulls=True).over(window)
)

# id_date is now in the format of yyyy-MM-dd converting it back to yyyyMMdd
filled_df = filled_df.withColumn(
 "id_date", F.date_format(F.col("id_date"), "yyyyMMdd")
)



## fact_currency_exchange_rate_monthly

# calculating the monthly average rates + months first rate and last rate
cols_to_grp_by = ["id_year_month", "base_currency", "to_currency", "source_api"]

window = Window.partitionBy(*cols_to_grp_by).orderBy("id_date")


df_with_first_last = (
 df_base.withColumn("first_conversion_rate", F.first("conversion_rate").over(window)
 ).withColumn("last_conversion_rate", F.last("conversion_rate").over(window))
)

df_monthly = (
 df_with_first_last
 .groupBy("id_year_month", "base_currency", "to_currency", "source_api")
 .agg(
        F.round(F.avg("conversion_rate"), 4).alias("avg_conversion_rate"),
        F.first("first_conversion_rate").alias("first_conversion_rate"),
        F.last("last_conversion_rate").alias("last_conversion_rate")
 )
)

# helper columns to calculate the YTD averages
df_monthly = df_monthly.withColumn("year", F.substring("id_year_month", 1, 4)) 

df_monthly = df_monthly.withColumn("id_year_month_int", F.col("id_year_month").cast("int"))



ytd_window = Window.partitionBy("year", "base_currency", "to_currency", "source_api"
 ).orderBy("id_year_month_int"
 ).rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_monthly
 = df_monthly.withColumn(
 "ytd_avg_conversion_rate",
 F.round(F.avg("avg_conversion_rate").over(ytd_window), 10)
).drop(*["year", "id_year_month_int"]
)